# Lab 20 — Multi-Agent Research Demo Notebook

Notebook này giúp bạn **thử nghiệm nhanh** các khối logic của bài lab trước khi implement chính thức trong `src/`.

**Luồng làm việc:**
1. Khám phá schemas & shared state
2. Mock services (LLM + Search) để chạy không cần API key
3. Viết các agent demo (Researcher → Analyst → Writer)
4. Supervisor routing + vòng lặp workflow mini
5. Benchmark single-agent vs multi-agent

> ⚠️ **Quy tắc:** Notebook chỉ để prototype. Sau khi chạy được ở đây, bạn phải **chuyển logic vào `src/multi_agent_research_lab/`** và pass tests. Các ô có `TODO(student)` là phần bạn phải tự viết.

## 0. Setup

Chạy từ repo root với package đã cài (`pip install -e ".[dev]"`).

In [ ]:
import sys
from pathlib import Path

# Cho phép import package khi chạy notebook từ thư mục notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

from multi_agent_research_lab.core.errors import StudentTodoError
from multi_agent_research_lab.core.schemas import (
    AgentName,
    AgentResult,
    BenchmarkMetrics,
    ResearchQuery,
    SourceDocument,
)
from multi_agent_research_lab.core.state import ResearchState

print("✅ Import OK — package sẵn sàng")

## 1. Khám phá Shared State

`ResearchState` là **single source of truth** được truyền qua mọi agent. Mỗi agent đọc state, cập nhật, rồi trả lại.

In [ ]:
query = ResearchQuery(
    query="So sánh RAG và fine-tuning cho domain adaptation",
    max_sources=3,
)
state = ResearchState(request=query)

state.record_route("researcher")
state.add_trace_event("demo", {"note": "first route recorded"})

print("Iteration:", state.iteration)
print("Route history:", state.route_history)
print("Trace:", state.trace)

## 2. Mock Services

Để demo không cần API key, ta dùng mock. Trong bản chính thức (`src/services/`), bạn sẽ nối provider thật (OpenAI / Tavily...).

- `MockSearchClient`: **đã viết sẵn** làm mẫu.
- `MockLLMClient`: **TODO(student)** — bạn tự viết theo cùng pattern.

In [ ]:
from dataclasses import dataclass


class MockSearchClient:
    """Trả về nguồn giả lập — cùng interface với services.search_client.SearchClient."""

    _FAKE_DOCS = [
        SourceDocument(
            title="RAG vs Fine-tuning: A Practical Guide",
            url="https://example.com/rag-vs-ft",
            snippet="RAG phù hợp khi dữ liệu thay đổi thường xuyên; fine-tuning tốt cho style/format.",
        ),
        SourceDocument(
            title="Retrieval-Augmented Generation Survey",
            url="https://example.com/rag-survey",
            snippet="RAG giảm hallucination bằng cách grounding vào tài liệu ngoài.",
        ),
        SourceDocument(
            title="When to Fine-tune LLMs",
            url="https://example.com/when-finetune",
            snippet="Fine-tuning hiệu quả khi cần hành vi nhất quán và latency thấp.",
        ),
    ]

    def search(self, query: str, max_results: int = 5) -> list[SourceDocument]:
        return self._FAKE_DOCS[:max_results]


@dataclass(frozen=True)
class MockLLMResponse:
    content: str
    input_tokens: int | None = None
    output_tokens: int | None = None


class MockLLMClient:
    """Giả lập LLM — cùng interface với services.llm_client.LLMClient."""

    def complete(self, system_prompt: str, user_prompt: str) -> MockLLMResponse:
        role = "analyst" if "analyst" in system_prompt.lower() else (
            "writer" if "writer" in system_prompt.lower() else "assistant"
        )
        context = [line.strip("- ") for line in user_prompt.splitlines() if len(line.strip()) > 30]
        bullets = "\n".join(f"- {line}" for line in context[:5]) or "- (không có ngữ cảnh)"

        if role == "analyst":
            content = (
                "Key claims:\n" + bullets + "\n"
                "Agreement: các nguồn thống nhất RAG hợp với dữ liệu thay đổi nhanh.\n"
                "Weak evidence: chưa có số liệu benchmark cụ thể."
            )
        elif role == "writer":
            content = "Tổng hợp:\n" + bullets + "\nTrade-off: RAG tốn retrieval, fine-tune tốn train."
        else:
            content = "Trả lời nhanh:\n" + bullets

        return MockLLMResponse(
            content=content,
            input_tokens=len(system_prompt + user_prompt) // 4,
            output_tokens=len(content) // 4,
        )


# Smoke test phần đã cho sẵn
search_client = MockSearchClient()
docs = search_client.search(query.query, max_results=query.max_sources)
for d in docs:
    print(f"- {d.title}: {d.snippet[:60]}...")

print(MockLLMClient().complete("You are an analyst.", "\n".join(d.snippet for d in docs)).content)


## 3. Demo Agents

Mỗi agent tuân theo contract `BaseAgent.run(state) -> state`.

- `DemoResearcherAgent`: **đã viết sẵn** làm mẫu — gọi search, ghi `sources` + `research_notes`.
- `DemoAnalystAgent`: **TODO(student)** — tổng hợp `sources` thành `analysis_notes`.
- `DemoWriterAgent`: **TODO(student)** — viết `final_answer` kèm citation.

In [ ]:
class DemoResearcherAgent:
    """MẪU: thu thập nguồn và ghi chú nghiên cứu."""

    name = "researcher"

    def __init__(self, search_client: MockSearchClient) -> None:
        self.search_client = search_client

    def run(self, state: ResearchState) -> ResearchState:
        docs = self.search_client.search(
            state.request.query, max_results=state.request.max_sources
        )
        state.sources = docs
        state.research_notes = "\n".join(f"- {d.title}: {d.snippet}" for d in docs)
        state.agent_results.append(
            AgentResult(
                agent=AgentName.RESEARCHER,
                content=state.research_notes,
                metadata={"num_sources": len(docs)},
            )
        )
        state.add_trace_event("researcher.done", {"num_sources": len(docs)})
        return state


class DemoAnalystAgent:
    """Phân tích sources thành analysis_notes."""

    name = "analyst"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        if not state.sources:
            state.errors.append("analyst: không có sources để phân tích")
            return state

        context = "\n".join(
            f"[{i}] {d.title}: {d.snippet}" for i, d in enumerate(state.sources, start=1)
        )
        response = self.llm_client.complete(
            system_prompt="You are an analyst. So sánh quan điểm và đánh dấu bằng chứng yếu.",
            user_prompt=f"Câu hỏi: {state.request.query}\n\n{context}",
        )
        state.analysis_notes = response.content
        state.agent_results.append(
            AgentResult(
                agent=AgentName.ANALYST,
                content=response.content,
                metadata={
                    "input_tokens": response.input_tokens,
                    "output_tokens": response.output_tokens,
                },
            )
        )
        state.add_trace_event("analyst.done", {"chars": len(response.content)})
        return state


class DemoWriterAgent:
    """Viết final_answer có trích dẫn nguồn."""

    name = "writer"

    def __init__(self, llm_client: MockLLMClient) -> None:
        self.llm_client = llm_client

    def run(self, state: ResearchState) -> ResearchState:
        context = state.analysis_notes or state.research_notes
        if not context:
            state.errors.append("writer: không có ngữ cảnh để viết")
            return state

        response = self.llm_client.complete(
            system_prompt="You are a writer. Viết câu trả lời cuối kèm citation [n].",
            user_prompt=(
                f"Câu hỏi: {state.request.query}\n"
                f"Đối tượng đọc: {state.request.audience}\n\n{context}"
            ),
        )
        citations = "\n".join(
            f"[{i}] {d.title} ({d.url})" for i, d in enumerate(state.sources, start=1)
        )
        state.final_answer = f"{response.content}\n\nNguồn:\n{citations}"
        state.agent_results.append(
            AgentResult(
                agent=AgentName.WRITER,
                content=state.final_answer,
                metadata={
                    "input_tokens": response.input_tokens,
                    "output_tokens": response.output_tokens,
                    "num_citations": len(state.sources),
                },
            )
        )
        state.add_trace_event("writer.done", {"chars": len(state.final_answer)})
        return state


# Smoke test agent mẫu
state = ResearchState(request=query)
state = DemoResearcherAgent(search_client).run(state)
print(state.research_notes)


## 4. Supervisor Routing

Supervisor quyết định agent nào chạy tiếp dựa trên state hiện tại. Đây là **trái tim của bài lab** — bạn tự thiết kế policy.

In [ ]:
MAX_ITERATIONS = 6
MAX_AGENT_RETRIES = 2


def demo_supervisor_route(state: ResearchState) -> str:
    """Trả về một trong: 'researcher' | 'analyst' | 'writer' | 'done'."""
    # Guard chống vòng lặp vô hạn — GIỮ NGUYÊN dòng này
    if state.iteration >= MAX_ITERATIONS:
        return "done"

    if state.final_answer:
        return "done"

    # Fallback: agent nào lỗi quá số lần cho phép thì bỏ qua, không retry mãi.
    def failures(agent: str) -> int:
        return sum(1 for err in state.errors if err.startswith(f"{agent}:"))

    if not state.sources and failures("researcher") < MAX_AGENT_RETRIES:
        return "researcher"
    if state.sources and not state.analysis_notes and failures("analyst") < MAX_AGENT_RETRIES:
        return "analyst"
    if (state.sources or state.research_notes) and failures("writer") < MAX_AGENT_RETRIES:
        return "writer"
    return "done"


print(demo_supervisor_route(ResearchState(request=query)))


## 5. Mini Workflow Loop

Vòng lặp điều phối **đã viết sẵn** — chỉ chạy được sau khi bạn hoàn thành các TODO ở trên. Đây chính là logic bạn sẽ chuyển thành LangGraph nodes/edges trong `graph/workflow.py`.

In [ ]:
def run_demo_workflow(query_text: str) -> ResearchState:
    q = ResearchQuery(query=query_text, max_sources=3)
    state = ResearchState(request=q)

    llm = MockLLMClient()
    agents = {
        "researcher": DemoResearcherAgent(MockSearchClient()),
        "analyst": DemoAnalystAgent(llm),
        "writer": DemoWriterAgent(llm),
    }

    while True:
        route = demo_supervisor_route(state)
        state.record_route(route)
        if route == "done":
            break
        state = agents[route].run(state)

    return state


try:
    final_state = run_demo_workflow("So sánh RAG và fine-tuning cho domain adaptation")
    print("Route history:", final_state.route_history)
    print("\n=== FINAL ANSWER ===\n")
    print(final_state.final_answer)
except StudentTodoError as exc:
    print(f"⛔ Còn TODO chưa hoàn thành: {exc}")
    print("→ Quay lại các ô trên, implement xong rồi chạy lại ô này.")

## 6. Benchmark: Single-agent vs Multi-agent

Dùng `run_benchmark` từ package để so sánh. Baseline single-agent (1 lần gọi LLM, không search) **bạn tự viết**.

In [ ]:
from multi_agent_research_lab.evaluation.benchmark import run_benchmark


def run_single_agent(query_text: str) -> ResearchState:
    """Baseline: một lần gọi LLM duy nhất, không search, không phân tích."""
    q = ResearchQuery(query=query_text, max_sources=3)
    state = ResearchState(request=q)

    response = MockLLMClient().complete(
        system_prompt="You are a single-agent assistant. Trả lời trực tiếp câu hỏi.",
        user_prompt=f"Câu hỏi: {query_text}\nĐối tượng đọc: {q.audience}",
    )
    state.final_answer = response.content
    state.record_route("single_agent")
    state.agent_results.append(
        AgentResult(
            agent=AgentName.WRITER,
            content=response.content,
            metadata={
                "input_tokens": response.input_tokens,
                "output_tokens": response.output_tokens,
            },
        )
    )
    return state


def compute_citation_coverage(state: ResearchState) -> float:
    """Tỷ lệ nguồn trong state.sources được nhắc đến trong final_answer."""
    answer = state.final_answer or ""
    if not state.sources or not answer:
        return 0.0

    hits = sum(
        1 for d in state.sources if d.title in answer or (d.url and d.url in answer)
    )
    return hits / len(state.sources)


demo_query = "So sánh RAG và fine-tuning cho domain adaptation"

try:
    results: list[BenchmarkMetrics] = []
    for run_name, runner in [
        ("single_agent", run_single_agent),
        ("multi_agent", run_demo_workflow),
    ]:
        st, metrics = run_benchmark(run_name, demo_query, runner)
        metrics.citation_coverage = compute_citation_coverage(st)
        results.append(metrics)

    print(f"{'run':<15}{'latency (s)':<15}{'citation cov.':<15}")
    for m in results:
        print(f"{m.run_name:<15}{m.latency_seconds:<15.3f}{m.citation_coverage!s:<15}")
except StudentTodoError as exc:
    print(f"⛔ Còn TODO chưa hoàn thành: {exc}")


## 7. Next Steps — đối chiếu với `src/`

Notebook là bản prototype. Bản chính thức đã được implement trong package:

| Notebook | Đích trong `src/multi_agent_research_lab/` |
|---|---|
| `MockLLMClient` → provider thật | `services/llm_client.py` (OpenAI + offline backend) |
| `MockSearchClient` → provider thật | `services/search_client.py` (Tavily + mock corpus) |
| `DemoResearcherAgent` / `DemoAnalystAgent` / `DemoWriterAgent` | `agents/researcher.py`, `agents/analyst.py`, `agents/writer.py` |
| `demo_supervisor_route` | `agents/supervisor.py` (`SupervisorAgent.decide`) |
| `run_demo_workflow` → LangGraph nodes/edges | `graph/workflow.py` |
| `compute_citation_coverage` + quality score | `evaluation/benchmark.py`, `agents/critic.py` |

Verify bản chính thức:
```bash
make lint && make test
python -m multi_agent_research_lab.cli baseline --query "..."
python -m multi_agent_research_lab.cli multi-agent --query "..." --trace-out reports/trace.json
python -m multi_agent_research_lab.cli benchmark
bash scripts/check_todos.sh   # không còn TODO(student) trong src/
```
